### Imports and source path

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

current_dir = Path.cwd()

possible_paths = [
    current_dir / "data" / "raw" / "online_retail_II.xlsx",
    current_dir.parent / "data" / "raw" / "online_retail_II.xlsx",
]

DATA_PATH = None

for path in possible_paths:
    if path.exists():
        DATA_PATH = path.resolve()
        break

print("Dataset:", DATA_PATH)

Dataset: C:\New folder\Hands on Projects\Data-Analysis\customer-sales-inventory-operations-optimization\data\raw\online_retail_II.xlsx


### Load and combine both worksheets

In [2]:
excel_file = pd.ExcelFile(DATA_PATH)

frames = []

for sheet in excel_file.sheet_names:
    df = pd.read_excel(DATA_PATH, sheet_name=sheet)
    
    df["SourceSheet"] = sheet
    
    frames.append(df)

raw_df = pd.concat(
    frames,
    ignore_index=True
)

print(f"Combined rows: {len(raw_df):,}")
print(f"Combined columns: {raw_df.shape[1]}")

display(raw_df.head())

Combined rows: 1,067,371
Combined columns: 9


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom,Year 2009-2010


### Basic structure

In [3]:
print("Dataset shape:", raw_df.shape)

print("\nColumns:")
for col in raw_df.columns:
    print("-", col)

print("\nData types:")
print(raw_df.dtypes)

Dataset shape: (1067371, 9)

Columns:
- Invoice
- StockCode
- Description
- Quantity
- InvoiceDate
- Price
- Customer ID
- Country
- SourceSheet

Data types:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
SourceSheet               str
dtype: object


### Missing-value profiling

In [4]:
missing_summary = pd.DataFrame({
    "Missing_Count": raw_df.isna().sum(),
    "Missing_Percentage": (
        raw_df.isna().mean() * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    "Missing_Count",
    ascending=False
)

display(missing_summary)

,Missing_Count,Missing_Percentage
Customer ID,243007,22.77
Description,4382,0.41
Invoice,0,0.00
Quantity,0,0.00
StockCode,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Country,0,0.00
SourceSheet,0,0.00


### Duplicate transactions

In [5]:
duplicate_count = raw_df.duplicated().sum()

duplicate_pct = (
    duplicate_count / len(raw_df)
) * 100

print(f"Exact duplicate rows: {duplicate_count:,}")
print(f"Duplicate percentage: {duplicate_pct:.2f}%")

Exact duplicate rows: 12,133
Duplicate percentage: 1.14%


In [6]:
duplicate_samples = raw_df[
    raw_df.duplicated(keep=False)
].sort_values(
    ["Invoice", "StockCode", "InvoiceDate"]
)

display(duplicate_samples.head(20))

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,"16,329.00",United Kingdom,Year 2009-2010
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,"16,329.00",United Kingdom,Year 2009-2010
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,Year 2009-2010
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,Year 2009-2010
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,Year 2009-2010
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,Year 2009-2010
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,Year 2009-2010
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,Year 2009-2010
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,Year 2009-2010
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,"16,329.00",United Kingdom,Year 2009-2010


### Unique business entities

In [7]:
print(
    "Unique invoices:",
    f"{raw_df['Invoice'].nunique():,}"
)

print(
    "Unique stock codes:",
    f"{raw_df['StockCode'].nunique():,}"
)

print(
    "Unique customers:",
    f"{raw_df['Customer ID'].nunique():,}"
)

print(
    "Unique countries:",
    f"{raw_df['Country'].nunique():,}"
)

Unique invoices: 53,628
Unique stock codes: 5,305
Unique customers: 5,942
Unique countries: 43


### Cancellation invoices

In [8]:
invoice_text = raw_df["Invoice"].astype(str)

cancellation_mask = invoice_text.str.startswith(
    "C",
    na=False
)

cancelled_rows = cancellation_mask.sum()

cancelled_invoices = raw_df.loc[
    cancellation_mask,
    "Invoice"
].nunique()

print(
    f"Cancellation transaction rows: "
    f"{cancelled_rows:,}"
)

print(
    f"Unique cancellation invoices: "
    f"{cancelled_invoices:,}"
)

print(
    f"Cancellation row percentage: "
    f"{cancelled_rows / len(raw_df) * 100:.2f}%"
)

Cancellation transaction rows: 19,494
Unique cancellation invoices: 8,292
Cancellation row percentage: 1.83%


In [9]:
display(
    raw_df.loc[cancellation_mask].head(15)
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,"16,321.00",Australia,Year 2009-2010
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,"16,321.00",Australia,Year 2009-2010
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,"16,321.00",Australia,Year 2009-2010
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,"16,321.00",Australia,Year 2009-2010
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,"16,321.00",Australia,Year 2009-2010
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,"16,321.00",Australia,Year 2009-2010
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,"16,321.00",Australia,Year 2009-2010
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,"16,321.00",Australia,Year 2009-2010
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,"16,321.00",Australia,Year 2009-2010
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,"17,592.00",United Kingdom,Year 2009-2010


### Quantity profiling

In [10]:
quantity_summary = raw_df["Quantity"].describe()

display(quantity_summary)

count   1,067,371.00
mean            9.94
std           172.71
min       -80,995.00
25%             1.00
50%             3.00
75%            10.00
max        80,995.00
Name: Quantity, dtype: float64

In [11]:
negative_quantity = (
    raw_df["Quantity"] < 0
).sum()

zero_quantity = (
    raw_df["Quantity"] == 0
).sum()

positive_quantity = (
    raw_df["Quantity"] > 0
).sum()

print(
    f"Negative quantities: "
    f"{negative_quantity:,}"
)

print(
    f"Zero quantities: "
    f"{zero_quantity:,}"
)

print(
    f"Positive quantities: "
    f"{positive_quantity:,}"
)

Negative quantities: 22,950
Zero quantities: 0
Positive quantities: 1,044,421


In [12]:
display(
    raw_df[
        raw_df["Quantity"] < 0
    ].head(20)
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,"16,321.00",Australia,Year 2009-2010
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,"16,321.00",Australia,Year 2009-2010
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,"16,321.00",Australia,Year 2009-2010
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,"16,321.00",Australia,Year 2009-2010
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,"16,321.00",Australia,Year 2009-2010
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,"16,321.00",Australia,Year 2009-2010
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,"16,321.00",Australia,Year 2009-2010
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,"16,321.00",Australia,Year 2009-2010
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,"16,321.00",Australia,Year 2009-2010
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,"17,592.00",United Kingdom,Year 2009-2010


### Price profiling

In [13]:
display(
    raw_df["Price"].describe()
)

count   1,067,371.00
mean            4.65
std           123.55
min       -53,594.36
25%             1.25
50%             2.10
75%             4.15
max        38,970.00
Name: Price, dtype: float64

In [14]:
negative_price = (
    raw_df["Price"] < 0
).sum()

zero_price = (
    raw_df["Price"] == 0
).sum()

positive_price = (
    raw_df["Price"] > 0
).sum()

print(
    f"Negative prices: "
    f"{negative_price:,}"
)

print(
    f"Zero prices: "
    f"{zero_price:,}"
)

print(
    f"Positive prices: "
    f"{positive_price:,}"
)

Negative prices: 5
Zero prices: 6,202
Positive prices: 1,061,164


In [15]:
display(
    raw_df[
        raw_df["Price"] <= 0
    ].head(30)
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom,Year 2009-2010
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom,Year 2009-2010
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom,Year 2009-2010
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom,Year 2009-2010
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.00,NaN,United Kingdom,Year 2009-2010
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.00,NaN,United Kingdom,Year 2009-2010
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.00,NaN,United Kingdom,Year 2009-2010
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.00,NaN,United Kingdom,Year 2009-2010
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.00,NaN,United Kingdom,Year 2009-2010
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.00,NaN,United Kingdom,Year 2009-2010


### Missing Customer IDs

In [16]:
missing_customer = (
    raw_df["Customer ID"].isna()
)

print(
    f"Rows missing Customer ID: "
    f"{missing_customer.sum():,}"
)

print(
    f"Percentage missing Customer ID: "
    f"{missing_customer.mean() * 100:.2f}%"
)

Rows missing Customer ID: 243,007
Percentage missing Customer ID: 22.77%


In [17]:
display(
    raw_df[
        missing_customer
    ].head(20)
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom,Year 2009-2010
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom,Year 2009-2010
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom,Year 2009-2010
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom,Year 2009-2010
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom,Year 2009-2010
578,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom,Year 2009-2010
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom,Year 2009-2010
1056,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom,Year 2009-2010
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom,Year 2009-2010
1058,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,NaN,United Kingdom,Year 2009-2010


### Missing descriptions

In [18]:
missing_description = (
    raw_df["Description"].isna()
)

print(
    f"Missing descriptions: "
    f"{missing_description.sum():,}"
)

print(
    f"Missing description percentage: "
    f"{missing_description.mean() * 100:.2f}%"
)

Missing descriptions: 4,382
Missing description percentage: 0.41%


In [19]:
display(
    raw_df[
        missing_description
    ].head(20)
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom,Year 2009-2010
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.00,NaN,United Kingdom,Year 2009-2010
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.00,NaN,United Kingdom,Year 2009-2010
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.00,NaN,United Kingdom,Year 2009-2010
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.00,NaN,United Kingdom,Year 2009-2010
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.00,NaN,United Kingdom,Year 2009-2010
6378,489882,35751C,NaN,12,2009-12-02 16:22:00,0.00,NaN,United Kingdom,Year 2009-2010
6555,489898,79323G,NaN,954,2009-12-03 09:40:00,0.00,NaN,United Kingdom,Year 2009-2010
6576,489901,21098,NaN,-200,2009-12-03 09:47:00,0.00,NaN,United Kingdom,Year 2009-2010
6581,489903,21166,NaN,48,2009-12-03 09:57:00,0.00,NaN,United Kingdom,Year 2009-2010


### Cancellation vs quantity relationship

In [20]:
raw_df["IsCancellation"] = (
    raw_df["Invoice"]
    .astype(str)
    .str.startswith("C", na=False)
)

raw_df["QuantitySign"] = np.select(
    [
        raw_df["Quantity"] < 0,
        raw_df["Quantity"] == 0,
        raw_df["Quantity"] > 0
    ],
    [
        "Negative",
        "Zero",
        "Positive"
    ],
    default="Unknown"
)

cancellation_quantity_check = pd.crosstab(
    raw_df["IsCancellation"],
    raw_df["QuantitySign"],
    margins=True
)

display(cancellation_quantity_check)

QuantitySign,Negative,Positive,All
IsCancellation,,,
False,3457,1044420,1047877
True,19493,1,19494
All,22950,1044421,1067371


### Negative quantities that are NOT cancellations

In [21]:
negative_non_cancel = raw_df[
    (raw_df["Quantity"] < 0) &
    (~raw_df["IsCancellation"])
]

print(
    f"Negative quantity rows without C invoice: "
    f"{len(negative_non_cancel):,}"
)

display(
    negative_non_cancel.head(30)
)

Negative quantity rows without C invoice: 3,457


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,IsCancellation,QuantitySign
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom,Year 2009-2010,False,Negative
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom,Year 2009-2010,False,Negative
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom,Year 2009-2010,False,Negative
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom,Year 2009-2010,False,Negative
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.00,NaN,United Kingdom,Year 2009-2010,False,Negative
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.00,NaN,United Kingdom,Year 2009-2010,False,Negative
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.00,NaN,United Kingdom,Year 2009-2010,False,Negative
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.00,NaN,United Kingdom,Year 2009-2010,False,Negative
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.00,NaN,United Kingdom,Year 2009-2010,False,Negative
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.00,NaN,United Kingdom,Year 2009-2010,False,Negative


### What descriptions appear in these transactions?

In [22]:
negative_non_cancel_summary = (
    negative_non_cancel
    .groupby(
        ["StockCode", "Description"],
        dropna=False
    )
    .agg(
        Rows=("Invoice", "size"),
        Total_Quantity=("Quantity", "sum"),
        Avg_Price=("Price", "mean")
    )
    .sort_values(
        "Rows",
        ascending=False
    )
)

display(
    negative_non_cancel_summary.head(30)
)

,,Rows,Total_Quantity,Avg_Price
StockCode,Description,,,
20852,given away,4,-10000,0.00
84559D,NaN,4,-124,0.00
20966,NaN,4,-344,0.00
21040,NaN,4,-75,0.00
85126,NaN,4,-42,0.00
85017A,NaN,4,-31,0.00
20892,NaN,4,-66,0.00
21161,NaN,4,-288,0.00
37464,NaN,4,-240,0.00


### Negative price investigation

In [23]:
negative_price_rows = raw_df[
    raw_df["Price"] < 0
]

print(
    f"Negative price rows: "
    f"{len(negative_price_rows):,}"
)

display(
    negative_price_rows
)

Negative price rows: 5


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,IsCancellation,QuantitySign
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,"-53,594.36",NaN,United Kingdom,Year 2009-2010,False,Positive
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,"-44,031.79",NaN,United Kingdom,Year 2009-2010,False,Positive
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,"-38,925.87",NaN,United Kingdom,Year 2009-2010,False,Positive
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,"-11,062.06",NaN,United Kingdom,Year 2010-2011,False,Positive
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,"-11,062.06",NaN,United Kingdom,Year 2010-2011,False,Positive


### Zero-price investigation

In [24]:
zero_price_rows = raw_df[
    raw_df["Price"] == 0
]

print(
    f"Zero-price rows: "
    f"{len(zero_price_rows):,}"
)

print(
    "Zero-price rows missing Customer ID:",
    f"{zero_price_rows['Customer ID'].isna().sum():,}"
)

print(
    "Zero-price rows missing Description:",
    f"{zero_price_rows['Description'].isna().sum():,}"
)

print(
    "Zero-price cancellation rows:",
    f"{zero_price_rows['IsCancellation'].sum():,}"
)

print(
    "Zero-price negative quantity rows:",
    f"{(zero_price_rows['Quantity'] < 0).sum():,}"
)

Zero-price rows: 6,202
Zero-price rows missing Customer ID: 6,131
Zero-price rows missing Description: 4,382
Zero-price cancellation rows: 0
Zero-price negative quantity rows: 3,457


In [25]:
display(
    zero_price_rows.head(30)
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,IsCancellation,QuantitySign
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom,Year 2009-2010,False,Negative
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom,Year 2009-2010,False,Negative
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom,Year 2009-2010,False,Negative
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom,Year 2009-2010,False,Negative
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.00,NaN,United Kingdom,Year 2009-2010,False,Negative
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.00,NaN,United Kingdom,Year 2009-2010,False,Positive
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.00,NaN,United Kingdom,Year 2009-2010,False,Negative
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.00,NaN,United Kingdom,Year 2009-2010,False,Negative
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.00,NaN,United Kingdom,Year 2009-2010,False,Positive
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.00,NaN,United Kingdom,Year 2009-2010,False,Negative


### Most common descriptions among zero-price records

In [26]:
zero_price_description_summary = (
    zero_price_rows["Description"]
    .fillna("[MISSING DESCRIPTION]")
    .value_counts()
    .head(30)
)

display(
    zero_price_description_summary
)

Description
[MISSING DESCRIPTION]                  4382
check                                   162
?                                        92
damages                                  84
damaged                                  81
found                                    28
missing                                  27
sold as set on dotcom                    20
Damaged                                  17
adjustment                               16
OWL DOORSTOP                             15
POLYESTER FILLER PAD 45x45cm             12
dotcom                                   12
amazon                                   11
POLYESTER FILLER PAD 40x40cm             10
IVORY KITCHEN SCALES                     10
FRENCH BLUE METAL DOOR SIGN 1            10
smashed                                   9
Found                                     9
PICNIC BASKET WICKER LARGE                9
AIRLINE BAG VINTAGE WORLD CHAMPION        9
BOX OF 24 COCKTAIL PARASOLS               9
RED KITCHEN SCALES  

### Missing descriptions relationship

In [27]:
missing_desc_df = raw_df[
    raw_df["Description"].isna()
]

print(
    "Missing description rows:",
    f"{len(missing_desc_df):,}"
)

print(
    "With Price = 0:",
    f"{(missing_desc_df['Price'] == 0).sum():,}"
)

print(
    "With Price > 0:",
    f"{(missing_desc_df['Price'] > 0).sum():,}"
)

print(
    "With Price < 0:",
    f"{(missing_desc_df['Price'] < 0).sum():,}"
)

print(
    "Missing Customer ID:",
    f"{missing_desc_df['Customer ID'].isna().sum():,}"
)

print(
    "Negative Quantity:",
    f"{(missing_desc_df['Quantity'] < 0).sum():,}"
)

Missing description rows: 4,382
With Price = 0: 4,382
With Price > 0: 0
With Price < 0: 0
Missing Customer ID: 4,382
Negative Quantity: 2,689


### Investigate unusual StockCodes

In [28]:
stockcode_text = (
    raw_df["StockCode"]
    .astype(str)
    .str.strip()
)

unusual_stockcode_mask = (
    ~stockcode_text.str.match(
        r"^\d{5}[A-Za-z]?$",
        na=False
    )
)

unusual_stockcodes = raw_df[
    unusual_stockcode_mask
]

print(
    "Rows with non-standard StockCodes:",
    f"{len(unusual_stockcodes):,}"
)

print(
    "Unique non-standard StockCodes:",
    f"{unusual_stockcodes['StockCode'].nunique():,}"
)

Rows with non-standard StockCodes: 7,465
Unique non-standard StockCodes: 67


In [29]:
unusual_stock_summary = (
    unusual_stockcodes
    .groupby(
        ["StockCode", "Description"],
        dropna=False
    )
    .agg(
        Rows=("Invoice", "size"),
        Quantity=("Quantity", "sum"),
        Avg_Price=("Price", "mean")
    )
    .sort_values(
        "Rows",
        ascending=False
    )
)

display(
    unusual_stock_summary.head(50)
)

,,Rows,Quantity,Avg_Price
StockCode,Description,,,
POST,POSTAGE,2115,5158,32.73
DOT,DOTCOM POSTAGE,1444,1438,223.45
M,Manual,1421,4607,528.78
15056BL,EDWARDIAN PARASOL BLACK,922,9433,6.23
C2,CARRIAGE,279,266,49.99
79323LP,LIGHT PINK CHERRY LIGHTS,230,975,6.73
D,Discount,177,-2872,72.87
79323GR,GREEN CHERRY LIGHTS,120,338,6.90
S,SAMPLES,104,-98,60.96


### Extreme quantity transactions

In [30]:
quantity_extremes = raw_df[
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country"
    ]
].copy()

display(
    quantity_extremes
    .sort_values(
        "Quantity",
        ascending=False
    )
    .head(20)
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,"16,446.00",United Kingdom
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,"12,346.00",United Kingdom
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,2010-02-15 11:57:00,0.10,"13,902.00",Denmark
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,2010-03-17 13:09:00,0.10,"13,902.00",Denmark
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,2010-03-17 13:09:00,0.10,"13,902.00",Denmark
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,2010-03-17 13:09:00,0.10,"13,902.00",Denmark
1027583,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,2011-11-25 15:57:00,0.00,"13,256.00",United Kingdom
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,2010-03-17 13:09:00,0.10,"13,902.00",Denmark
192197,507637,84016,FLAG OF ST GEORGE CAR FLAG,10200,2010-05-10 14:55:00,0.00,NaN,United Kingdom
135027,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,2010-03-23 15:36:00,0.25,"17,940.00",United Kingdom


In [31]:
display(
    quantity_extremes
    .sort_values(
        "Quantity",
        ascending=True
    )
    .head(20)
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,"16,446.00",United Kingdom
587085,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,1.04,"12,346.00",United Kingdom
750991,556691,23005,printing smudges/thrown away,-9600,2011-06-14 10:37:00,0.00,NaN,United Kingdom
750990,556690,23005,printing smudges/thrown away,-9600,2011-06-14 10:37:00,0.00,NaN,United Kingdom
303996,519017,22759,NaN,-9600,2010-08-13 09:14:00,0.00,NaN,United Kingdom
529729,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,2010-12-02 14:23:00,0.03,"15,838.00",United Kingdom
507225,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,2010-12-02 14:23:00,0.03,"15,838.00",United Kingdom
156488,504311,22197,NaN,-9200,2010-04-12 14:39:00,0.00,NaN,United Kingdom
750989,556687,23003,Printing smudges/thrown away,-9058,2011-06-14 10:36:00,0.00,NaN,United Kingdom
194372,507913,10120,Zebra invcing error,-9000,2010-05-11 17:16:00,0.00,NaN,United Kingdom


### Extreme price transactions

In [32]:
display(
    raw_df[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country"
        ]
    ]
    .sort_values(
        "Price",
        ascending=False
    )
    .head(20)
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
748142,C556445,M,Manual,-1,2011-06-10 15:31:00,"38,970.00","15,098.00",United Kingdom
241824,C512770,M,Manual,-1,2010-06-17 16:52:00,"25,111.09","17,399.00",United Kingdom
241827,512771,M,Manual,1,2010-06-17 16:53:00,"25,111.09",NaN,United Kingdom
320581,C520667,BANK CHARGES,Bank Charges,-1,2010-08-27 13:42:00,"18,910.69",NaN,United Kingdom
1050063,C580605,AMAZONFEE,AMAZON FEE,-1,2011-12-05 11:36:00,"17,836.46",NaN,United Kingdom
569163,C540117,AMAZONFEE,AMAZON FEE,-1,2011-01-05 09:55:00,"16,888.02",NaN,United Kingdom
569164,C540118,AMAZONFEE,AMAZON FEE,-1,2011-01-05 09:57:00,"16,453.71",NaN,United Kingdom
517955,537632,AMAZONFEE,AMAZON FEE,1,2010-12-07 15:08:00,"13,541.33",NaN,United Kingdom
519294,C537651,AMAZONFEE,AMAZON FEE,-1,2010-12-07 15:49:00,"13,541.33",NaN,United Kingdom
540477,C537630,AMAZONFEE,AMAZON FEE,-1,2010-12-07 15:04:00,"13,541.33",NaN,United Kingdom


In [33]:
display(
    raw_df[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country"
        ]
    ]
    .sort_values(
        "Price",
        ascending=True
    )
    .head(20)
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,"-53,594.36",NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,"-44,031.79",NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,"-38,925.87",NaN,United Kingdom
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,"-11,062.06",NaN,United Kingdom
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,"-11,062.06",NaN,United Kingdom
80382,496757,37504,NaN,-2,2010-02-03 15:01:00,0.00,NaN,United Kingdom
318825,520492,22307,NaN,-200,2010-08-26 11:59:00,0.00,NaN,United Kingdom
318826,520490,22305,NaN,-200,2010-08-26 11:59:00,0.00,NaN,United Kingdom
318827,520491,22306,NaN,-200,2010-08-26 11:59:00,0.00,NaN,United Kingdom
80589,496791,21183,NaN,-2,2010-02-03 16:45:00,0.00,NaN,United Kingdom


### Are exact duplicates concentrated somewhere?

In [34]:
duplicate_rows = raw_df[
    raw_df.duplicated(
        subset=[
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country"
        ],
        keep=False
    )
]

print(
    f"Rows involved in duplicate groups: "
    f"{len(duplicate_rows):,}"
)

print(
    "Cancellation rows among duplicates:",
    f"{duplicate_rows['IsCancellation'].sum():,}"
)

print(
    "Missing customers among duplicates:",
    f"{duplicate_rows['Customer ID'].isna().sum():,}"
)

Rows involved in duplicate groups: 67,242
Cancellation rows among duplicates: 774
Missing customers among duplicates: 15,702


### Investigate the one positive cancellation

In [35]:
positive_cancellation = raw_df[
    (raw_df["IsCancellation"]) &
    (raw_df["Quantity"] > 0)
]

print(
    "Positive quantity cancellation rows:",
    len(positive_cancellation)
)

display(
    positive_cancellation[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country",
            "SourceSheet"
        ]
    ]
)

Positive quantity cancellation rows: 1


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
76799,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,NaN,United Kingdom,Year 2009-2010


### Zero-price rows that actually have customers

In [36]:
zero_price_customer = raw_df[
    (raw_df["Price"] == 0) &
    (raw_df["Customer ID"].notna())
].copy()

print(
    "Zero-price rows with Customer ID:",
    f"{len(zero_price_customer):,}"
)

display(
    zero_price_customer[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country"
        ]
    ].head(100)
)

Zero-price rows with Customer ID: 71


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.00,"16,126.00",United Kingdom
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.00,"15,658.00",United Kingdom
16107,490727,M,Manual,1,2009-12-07 16:38:00,0.00,"17,231.00",United Kingdom
18738,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,2009-12-08 15:25:00,0.00,"14,108.00",United Kingdom
18739,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-08 15:25:00,0.00,"14,108.00",United Kingdom
32916,492079,85042,ANTIQUE LILY FAIRY LIGHTS,8,2009-12-15 13:49:00,0.00,"15,070.00",United Kingdom
40101,492760,21143,ANTIQUE GLASS HEART DECORATION,12,2009-12-18 14:22:00,0.00,"18,071.00",United Kingdom
47126,493761,79320,FLAMINGO LIGHTS,24,2010-01-06 14:54:00,0.00,"14,258.00",United Kingdom
48342,493899,22355,"CHARLOTTE BAG , SUKI DESIGN",10,2010-01-08 10:43:00,0.00,"12,417.00",Belgium
57619,494607,21533,RETRO SPOT LARGE MILK JUG,12,2010-01-15 12:43:00,0.00,"16,858.00",United Kingdom


In [37]:
display(
    zero_price_customer["Description"]
    .value_counts(dropna=False)
    .head(30)
)

Description
Manual                               7
CHRISTMAS PUDDING TRINKET POT        2
This is a test product.              2
REGENCY CAKESTAND 3 TIER             2
ROUND CAKE TIN VINTAGE GREEN         2
6 RIBBONS EMPIRE                     1
DOOR MAT FAIRY CAKE                  1
CHRISTMAS CRAFT WHITE FAIRY          1
ANTIQUE LILY FAIRY LIGHTS            1
ANTIQUE GLASS HEART DECORATION       1
 FLAMINGO LIGHTS                     1
CHARLOTTE BAG , SUKI DESIGN          1
RETRO SPOT LARGE MILK JUG            1
VINTAGE GLASS COFFEE CADDY           1
CAST IRON HOOK GARDEN TROWEL         1
CAST IRON HOOK GARDEN FORK           1
AIRLINE BAG VINTAGE JET SET WHITE    1
HANGING METAL BIRD BATH              1
SET/5 RED SPOTTY LID GLASS BOWLS     1
DOORMAT HOME SWEET HOME BLUE         1
TV DINNER TRAY DOLLY GIRL            1
MILK PAN PINK RETROSPOT              1
POLYESTER FILLER PAD 45x45cm         1
CAKE STAND LACE WHITE                1
DOLLY GIRL LUNCH BOX                 1
NOEL WOODEN B

### Investigate known non-merchandise codes

In [38]:
candidate_operational_codes = [
    "POST",
    "DOT",
    "M",
    "C2",
    "S",
    "BANK CHARGES",
    "ADJUST",
    "ADJUST2",
    "D",
    "B"
]

operational_candidates = raw_df[
    raw_df["StockCode"]
    .astype(str)
    .str.strip()
    .isin(candidate_operational_codes)
]

operational_code_summary = (
    operational_candidates
    .groupby(
        ["StockCode", "Description"],
        dropna=False
    )
    .agg(
        Rows=("Invoice", "size"),
        Quantity=("Quantity", "sum"),
        Min_Price=("Price", "min"),
        Avg_Price=("Price", "mean"),
        Max_Price=("Price", "max")
    )
    .sort_values(
        "Rows",
        ascending=False
    )
)

display(operational_code_summary)

Rows  Quantity  Min_Price  \
StockCode    Description                                                      
POST         POSTAGE                              2115      5158       0.50   
DOT          DOTCOM POSTAGE                       1444      1438       0.00   
M            Manual                               1421      4607       0.00   
C2           CARRIAGE                              279       266      15.00   
D            Discount                              177     -2872       0.01   
S            SAMPLES                               104       -98       2.80   
BANK CHARGES Bank Charges                           96       -40       0.00   
ADJUST       Adjustment by john on 26/01/2010 16    38         2       4.57   
             Adjustment by john on 26/01/2010 17    26         6      10.51   
POST         NaN                                     7      4950       0.00   
B            Adjust bad debt                         6         6 -53,594.36   
BANK CHARGES  Bank Charges                           6        -2      15.00   
ADJUST       Adjustment by Peter on 24/05/2010 1     3        -3      72.45   
ADJUST2      Adjustment by Peter on Jun 25 2010      3         3      72.45   
C2           NaN                                     3       450       0.00   
DOT          NaN                                     2      1500       0.00   

                                                  Avg_Price  Max_Price  
StockCode    Description                                                
POST         POSTAGE                                  32.73   8,142.75  
DOT          DOTCOM POSTAGE                          223.45   4,505.17  
M            Manual                                  528.78  38,970.00  
C2           CARRIAGE                                 49.99     150.00  
D            Discount                                 72.87   1,867.86  
S            SAMPLES                                  60.96     605.18  
BANK CHARGES Bank Charges                            359.30  18,910.69  
ADJUST       Adjustment by john on 26/01/2010 16      75.41     342.80  
             Adjustment by john on 26/01/2010 17     283.23   5,117.03  
POST         NaN                                       0.00       0.00  
B            Adjust bad debt                     -24,602.35  11,062.06  
BANK CHARGES  Bank Charges                           354.83     848.43  
ADJUST       Adjustment by Peter on 24/05/2010 1     243.68     358.47  
ADJUST2      Adjustment by Peter on Jun 25 2010      243.68     358.47  
C2           NaN                                       0.00       0.00  
DOT          NaN                                       0.00       0.00

### Highest positive prices

In [39]:
highest_prices = (
    raw_df[
        raw_df["Price"] > 0
    ][
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country"
        ]
    ]
    .sort_values(
        "Price",
        ascending=False
    )
    .head(30)
)

display(highest_prices)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
748142,C556445,M,Manual,-1,2011-06-10 15:31:00,"38,970.00","15,098.00",United Kingdom
241827,512771,M,Manual,1,2010-06-17 16:53:00,"25,111.09",NaN,United Kingdom
241824,C512770,M,Manual,-1,2010-06-17 16:52:00,"25,111.09","17,399.00",United Kingdom
320581,C520667,BANK CHARGES,Bank Charges,-1,2010-08-27 13:42:00,"18,910.69",NaN,United Kingdom
1050063,C580605,AMAZONFEE,AMAZON FEE,-1,2011-12-05 11:36:00,"17,836.46",NaN,United Kingdom
569163,C540117,AMAZONFEE,AMAZON FEE,-1,2011-01-05 09:55:00,"16,888.02",NaN,United Kingdom
569164,C540118,AMAZONFEE,AMAZON FEE,-1,2011-01-05 09:57:00,"16,453.71",NaN,United Kingdom
517955,537632,AMAZONFEE,AMAZON FEE,1,2010-12-07 15:08:00,"13,541.33",NaN,United Kingdom
517953,C537630,AMAZONFEE,AMAZON FEE,-1,2010-12-07 15:04:00,"13,541.33",NaN,United Kingdom
540478,537632,AMAZONFEE,AMAZON FEE,1,2010-12-07 15:08:00,"13,541.33",NaN,United Kingdom
